# 03. Model Storage & Capacity Gate

Evaluates **authoritative Google Drive API account quota** before initiating the 142-shard model transfer into `/content/drive/MyDrive/AI - Google Drive/GLM-5.2/model`.

### Capacity Semantics
| Layer | Source | Role |
|---|---|---|
| **Google Drive Account Quota** | `drive.about.get()` API | **Authoritative gate** — must have >= 400 GB free |
| **Colab Local NVMe** | `shutil.disk_usage('/content')` | Temp buffer only — 3 GiB per chunk, never full model |
| **FUSE Mount Capacity** | `shutil.disk_usage(drive_mount_path)` | **Diagnostic only** — never used as a gate |

> **The full 399.79 GiB model is NOT staged locally.** It downloads directly to Google Drive.

### Step 1: Pre-Download Capacity Gate

In [ ]:
import os
import sys
import shutil
import subprocess

# Ensure repository is present and synchronized to latest commit
REPO_DIR = '/content/glm52-drive-runtime'
if not os.path.exists(REPO_DIR):
    print(f"Cloning GLM-5.2 repository into {REPO_DIR}...")
    subprocess.run(['git', 'clone', 'https://github.com/Aqib2607/AI.git', REPO_DIR], check=True)
else:
    print(f"Synchronizing {REPO_DIR} to latest master...")
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', 'master'], check=False)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from scripts.drive_check import get_drive_service, get_drive_storage_quota, evaluate_storage_gate
from scripts.download_model import get_capacity_report, evaluate_download_gate, is_drive_path

# ─── Model Specifications ────────────────────────────────────────────────────
MODEL_REPO           = 'mastouri/GLM-5.2-colibri-int4-g64-with-int8-mtp'
MODEL_SIZE_GIB       = 399.79
MODEL_SIZE_GB        = 429.28
REQUIRED_DRIVE_GB    = 400.0
RECOMMENDED_DRIVE_GB = 450.0
TEMP_CHUNK_GIB       = 3.0      # Per-chunk local NVMe buffer only

# ─── Paths ───────────────────────────────────────────────────────────────────
DRIVE_MODEL_DIR = '/content/drive/MyDrive/AI - Google Drive/GLM-5.2/model'
LOCAL_CONTENT   = '/content'

# ─── 1. Authoritative Google Drive API Quota ─────────────────────────────────
service  = get_drive_service()
capacity = get_capacity_report(DRIVE_MODEL_DIR, drive_service=service)

gate_status, gate_reason = evaluate_download_gate(
    capacity,
    required_gb=REQUIRED_DRIVE_GB,
    recommended_gb=RECOMMENDED_DRIVE_GB,
    local_temp_gib=TEMP_CHUNK_GIB
)

dq    = capacity['drive_api_quota']
local = capacity['local_colab_disk']
fuse  = capacity['fuse_diagnostic']

# ─── 2. Print Capacity Report ─────────────────────────────────────────────────
SEP = '=' * 75
sep = '-' * 75
print(SEP)
print('   GLM-5.2 COLIBRI  |  STORAGE CAPACITY PREFLIGHT REPORT')
print(SEP)
print(f"Model:                           {MODEL_REPO}")
print(f"Model Size:                      {MODEL_SIZE_GIB:.2f} GiB ({MODEL_SIZE_GB:.2f} GB decimal)")
print(f"Full model staged to local disk: NO — downloads directly to Google Drive")
print(f"Target path:                     {DRIVE_MODEL_DIR}")
print(f"Target is Drive mount:           {is_drive_path(DRIVE_MODEL_DIR)}")
print(sep)

print("SECTION 1 — Google Drive Account Quota (AUTHORITATIVE GATE)")
print(f"  Account:           {dq.get('email', 'aqibjawwad2607@gmail.com')}")
print(f"  Total Plan Quota:  {f\"{dq['limit_gb']:,.2f} GB\" if dq.get('limit_gb') else 'Unlimited'}")
print(f"  Used:              {f\"{dq['usage_gb']:,.2f} GB\" if dq.get('usage_gb') is not None else 'N/A'}")
print(f"  Available Free:    {f\"{dq['free_gb']:,.2f} GB\" if dq.get('free_gb') is not None else 'Unlimited' if dq.get('is_unlimited') else 'UNAVAILABLE'}")
print(f"  Required:          >= {REQUIRED_DRIVE_GB:.2f} GB  (Recommended: >= {RECOMMENDED_DRIVE_GB:.2f} GB)")
print(f"  GATE DECISION:     {gate_status}")
print(f"  Reason:            {gate_reason}")
print(sep)

print("SECTION 2 — Colab Local Ephemeral NVMe (TEMPORARY CHUNK BUFFER ONLY)")
print(f"  Local Free:        {local.get('free_gib', 'N/A')} GiB")
print(f"  Local Total:       {local.get('total_gib', 'N/A')} GiB")
print(f"  Required:          {TEMP_CHUNK_GIB:.2f} GiB (per-chunk temp only, not full model)")
print(f"  NOTE: The full {MODEL_SIZE_GIB:.2f} GiB model is NOT copied to /content.")
print(sep)

print("SECTION 3 — Drive FUSE Mount Diagnostics (INFORMATIONAL ONLY — NOT a gate)")
if fuse.get('free_gib') is not None:
    print(f"  FUSE Free:         {fuse['free_gib']:.2f} GiB  (Colab virtual container overlay)")
    print(f"  FUSE Total:        {fuse['total_gib']:.2f} GiB")
    print(f"  This value does NOT represent your Google Drive account storage capacity.")
else:
    print("  FUSE path not yet mounted or accessible.")
print(SEP)

# ─── 3. Final Decision ────────────────────────────────────────────────────────
# GO_WITH_RECOMMENDED_MARGIN means Drive API has >= 500 GB free — always a success.
# GO_WITH_LOW_MARGIN means Drive API has >= 400 GB but < 450 GB — still a pass.
# GO means Drive API has >= 450 GB and < 500 GB free — a pass.
# Only actual NO-GO / NO-GO_UNKNOWN_QUOTA states block the download.
is_go = gate_status in ('GO', 'GO_WITH_LOW_MARGIN', 'GO_WITH_RECOMMENDED_MARGIN')
print(f"PREFLIGHT DECISION: {'✓ GO — READY FOR DOWNLOAD' if is_go else '✗ NO-GO — INSUFFICIENT GOOGLE DRIVE QUOTA'}")
print(SEP)

if not is_go:
    raise SystemExit(f"Download blocked: {gate_reason}")
else:
    print("\n✓ Capacity gate passed. Proceed to Step 2 to start the download.")

### Step 2: Resumable Model Shard Download

Downloads directly to Google Drive. Preserves existing completed shards and resumes partial `.tmp` files automatically.

> **Authentication**: `mastouri/GLM-5.2-colibri-int4-g64-with-int8-mtp` is a public repository. An HF token is not required. If you have already set `HF_TOKEN` in your Colab environment (Secrets or a prior cell), it will be used automatically — do not overwrite it here.

In [ ]:
import os

# The target repository is public — no token is required.
# If HF_TOKEN is already set in your Colab Secrets or environment,
# it will be forwarded automatically. Do not overwrite it here.

DRIVE_MODEL_DIR = '/content/drive/MyDrive/AI - Google Drive/GLM-5.2/model'
MODEL_REPO      = 'mastouri/GLM-5.2-colibri-int4-g64-with-int8-mtp'

!python /content/glm52-drive-runtime/scripts/download_model.py \
  --repo "{MODEL_REPO}" \
  --target-dir "{DRIVE_MODEL_DIR}" \
  --required-gb 400 \
  --recommended-gb 450 \
  --local-temp-gib 3